# 3DGS Vanilla (mipnerf360)

mipnerf360 데이터셋에 포함된 COLMAP 결과를 그대로 사용하여 3DGS 학습 → 렌더링 → 평가를 수행합니다.

COLMAP을 직접 실행하지 않고, 데이터셋의 `sparse/0/`와 `images_N/`을 바로 사용합니다.

In [ ]:
# === 셀 1: 설정 ===
import os, glob, json
from PIL import Image

# ========== 여기만 수정 ==========
SCENE_NAME = "bicycle"
# 사용 가능한 씬: bicycle, bonsai, counter, garden, kitchen, room, stump

IMAGE_SCALE = 1  # 이미지 스케일 (1=원본, 2=1/2, 4=1/4, 8=1/8)
# ================================

BASE_DIR = "/home/daeho/storage/3dgs_sba"
GS_REPO = os.path.join(BASE_DIR, "repos", "gaussian-splatting")
DATASET_DIR = os.path.join(BASE_DIR, "datasets", "mipnerf360", SCENE_NAME)
OUTPUT_DIR = os.path.join(BASE_DIR, "output", f"{SCENE_NAME}_vanilla")

# 3DGS train.py는 -s 경로 아래에서 images/ 또는 images_N/ 을 자동으로 찾음
# --resolution N 을 지정하면 images_N/ 을 사용
SPARSE_DIR = os.path.join(DATASET_DIR, "sparse", "0")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"SCENE_NAME   : {SCENE_NAME}")
print(f"IMAGE_SCALE  : 1/{IMAGE_SCALE}" if IMAGE_SCALE > 1 else f"IMAGE_SCALE  : 원본")
print(f"DATASET_DIR  : {DATASET_DIR}")
print(f"SPARSE_DIR   : {SPARSE_DIR} ({'OK' if os.path.isdir(SPARSE_DIR) else 'NOT FOUND'})")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")

# 이미지 확인
if IMAGE_SCALE == 1:
    img_dir = os.path.join(DATASET_DIR, "images")
else:
    img_dir = os.path.join(DATASET_DIR, f"images_{IMAGE_SCALE}")

img_files = sorted(glob.glob(os.path.join(img_dir, "*")))
if img_files:
    sample = Image.open(img_files[0])
    print(f"\n이미지: {len(img_files)}장, {sample.size[0]}x{sample.size[1]} ({os.path.basename(img_dir)}/)")

In [ ]:
# === 셀 2: 3DGS 학습 ===
# --resolution N: 데이터셋의 images_N/ 폴더를 자동으로 사용
# -s: mipnerf360 데이터셋 경로 (images/, sparse/ 가 있는 곳)
%cd {GS_REPO}
!python train.py \
    -s {DATASET_DIR} \
    -m {OUTPUT_DIR} \
    --iterations 30000 \
    --densify_grad_threshold 0.001 \
    --resolution {IMAGE_SCALE} \
    --eval \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 15000 30000

In [ ]:
# === 셀 3: 렌더링 ===
%cd {GS_REPO}
!python render.py -m {OUTPUT_DIR} --iteration 30000

In [ ]:
# === 셀 4: 메트릭 평가 (PSNR / SSIM / LPIPS) ===
%cd {GS_REPO}
!python metrics.py -m {OUTPUT_DIR}

results_path = os.path.join(OUTPUT_DIR, "results.json")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\n=== 평가 결과 ===")
    for iteration, metrics in results.items():
        print(f"\n[{iteration}]")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")
    print("\n✅ 평가 완료")
else:
    print("❌ results.json 파일이 없습니다.")

In [ ]:
# === 셀 5: 렌더링 이미지 시각화 ===
import matplotlib.pyplot as plt

render_dir = os.path.join(OUTPUT_DIR, "test", "ours_30000", "renders")
gt_dir = os.path.join(OUTPUT_DIR, "test", "ours_30000", "gt")

if not os.path.isdir(render_dir):
    print(f"❌ 렌더링 결과가 없습니다: {render_dir}")
else:
    render_imgs = sorted(glob.glob(os.path.join(render_dir, "*.png")))
    gt_imgs = sorted(glob.glob(os.path.join(gt_dir, "*.png")))

    n_show = min(4, len(render_imgs))
    fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 10))
    if n_show == 1:
        axes = axes.reshape(2, 1)

    for i in range(n_show):
        render_img = Image.open(render_imgs[i])
        axes[0, i].imshow(render_img)
        axes[0, i].set_title(f"Render {i}")
        axes[0, i].axis("off")

        if i < len(gt_imgs):
            gt_img = Image.open(gt_imgs[i])
            axes[1, i].imshow(gt_img)
            axes[1, i].set_title(f"GT {i}")
            axes[1, i].axis("off")

    plt.suptitle(f"3DGS Vanilla - {SCENE_NAME} (1/{IMAGE_SCALE})", fontsize=16)
    plt.tight_layout()
    plt.show()
    print(f"✅ {len(render_imgs)}장 렌더링 이미지 중 {n_show}장 표시")